# ALPR Training Pipeline (Colab)

This notebook trains and evaluates both models:
- Plate detection model (YOLOv8)
- Plate character recognition model (YOLOv8)

It stores all outputs under `artifacts/`:
- `artifacts/detection/`
- `artifacts/recognition/`
- `artifacts/models/`
- `artifacts/eval_images/`

Run cells from top to bottom in Colab with GPU enabled.

In [1]:
# 1) Install dependencies (Colab)
# If you run locally and already installed deps, this is safe to keep.
%pip install -q -U pip
%pip install -q ultralytics opencv-python pyyaml onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.8 MB/s eta 0:00:00


In [2]:
# 2) Resolve project directory and create artifacts folders
from pathlib import Path
import os
import shutil
import subprocess
import sys

def is_project_dir(path: Path) -> bool:
    # Adjusting check to be more lenient: just needs to exist
    return path.exists() and (path / 'main.py').exists()

# 1) Explicit common locations
candidates = [
    Path('/content/Traffic-Serveillance-System/ai-service'),
    Path('/content/traffic surveillance system/ai-service'),
]

# 2) From current working directory and its parents
cwd = Path.cwd().resolve()
candidates.extend([cwd, *cwd.parents])

# 3) Colab fallback: search one level under /content
content_root = Path('/content')
if content_root.exists():
    for p in content_root.glob('*/ai-service'):
        candidates.append(p)

PROJECT_DIR = None
seen = set()
for p in candidates:
    p = p.resolve()
    if p in seen:
        continue
    seen.add(p)
    if is_project_dir(p):
        PROJECT_DIR = p
        break

# 4) Force clone if missing
if PROJECT_DIR is None and content_root.exists():
    repo_root = content_root / 'Traffic-Serveillance-System'
    if repo_root.exists():
        shutil.rmtree(repo_root)

    print('Cloning repository...')
    subprocess.run([
        'git', 'clone', '--depth', '1', 'https://github.com/kareemtaha3/Traffic-Serveillance-System.git', str(repo_root)
    ], check=True)

    fallback = repo_root / 'ai-service'
    if is_project_dir(fallback):
        PROJECT_DIR = fallback
    else:
        print(f'Contents of {repo_root}:', os.listdir(repo_root))
        if fallback.exists():
            print(f'Contents of {fallback}:', os.listdir(fallback))

if PROJECT_DIR is None:
    # Last ditch attempt: check if ai-service is the repo root itself
    if is_project_dir(Path('/content/Traffic-Serveillance-System')):
        PROJECT_DIR = Path('/content/Traffic-Serveillance-System')
    else:
        raise RuntimeError('Could not locate ai-service project directory. Check the output above for directory structure.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
DETECTION_ARTIFACTS = ARTIFACTS_DIR / 'detection'
RECOGNITION_ARTIFACTS = ARTIFACTS_DIR / 'recognition'
MODELS_ARTIFACTS = ARTIFACTS_DIR / 'models'
EVAL_IMAGES_DIR = ARTIFACTS_DIR / 'eval_images'

for d in [ARTIFACTS_DIR, DETECTION_ARTIFACTS, RECOGNITION_ARTIFACTS, MODELS_ARTIFACTS, EVAL_IMAGES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Artifacts folder ready:', ARTIFACTS_DIR)

Cloning repository...
PROJECT_DIR = /content/Traffic-Serveillance-System/ai-service
Artifacts folder ready: /content/Traffic-Serveillance-System/ai-service/artifacts


In [3]:
# 3) Get dataset (clone if missing)
DATASET_DIR = PROJECT_DIR / 'dataset'
if not DATASET_DIR.exists():
    subprocess.run([
        'git', 'clone', 'https://github.com/ahmedramadan96/EALPR.git', str(DATASET_DIR)
    ], check=True)
else:
    print('Dataset already exists:', DATASET_DIR)

# quick sanity check
for rel in [
    'EALPR Vechicles dataset/Vehicles',
    'EALPR Vechicles dataset/Vehicles Labeling',
    'EALPR- Plates dataset',
    'EALPR- LP characters dataset/Characters Labeling',
]:
    p = DATASET_DIR / rel
    print(rel, 'OK' if p.exists() else 'MISSING')

EALPR Vechicles dataset/Vehicles OK
EALPR Vechicles dataset/Vehicles Labeling OK
EALPR- Plates dataset OK
EALPR- LP characters dataset/Characters Labeling OK


In [4]:
# 4) Emergency Data Preparation (Custom Script)
import sys
import os
import shutil
from pathlib import Path

det_out = PROJECT_DIR / 'data/processed/detection'
rec_out = PROJECT_DIR / 'data/processed/recognition'

# Ensure output directories exist
for d in [det_out, rec_out]:
    (d / 'images' / 'train').mkdir(parents=True, exist_ok=True)
    (d / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (d / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
    (d / 'labels' / 'val').mkdir(parents=True, exist_ok=True)

def simple_prepare():
    print("Starting custom dataset preparation...")

    # Clean target directories before copying
    for d_type in ['images', 'labels']:
        for split in ['train', 'val']:
            det_target_dir = det_out / d_type / split
            if det_target_dir.exists():
                shutil.rmtree(det_target_dir)
            det_target_dir.mkdir(parents=True, exist_ok=True)

            rec_target_dir = rec_out / d_type / split
            if rec_target_dir.exists():
                shutil.rmtree(rec_target_dir)
            rec_target_dir.mkdir(parents=True, exist_ok=True)

    # --- Plate Detection Preparation ---
    det_images_src = DATASET_DIR / 'EALPR Vechicles dataset/Vehicles'
    det_labels_src = DATASET_DIR / 'EALPR Vechicles dataset/Vehicles Labeling'

    det_pairs = []
    for label_path in sorted(list(det_labels_src.glob('*.txt'))):
        stem = label_path.stem
        img_path_jpg = (det_images_src / stem).with_suffix('.jpg')
        img_path_png = (det_images_src / stem).with_suffix('.png')

        if img_path_jpg.exists():
            det_pairs.append((img_path_jpg, label_path))
        elif img_path_png.exists():
            det_pairs.append((img_path_png, label_path))
        # else:
        #     print(f"Warning: No image found for label {label_path.name} in {det_images_src}")

    print(f"Found {len(det_pairs)} image-label pairs for detection.")
    if not det_pairs:
        print("Error: No detection image-label pairs found. Check dataset paths, file existence, and naming conventions.")
        return

    # Split detection data
    split_idx_det = int(len(det_pairs) * 0.8)
    train_det_pairs = det_pairs[:split_idx_det]
    val_det_pairs = det_pairs[split_idx_det:]

    # Copy detection data
    for img_path, label_path in train_det_pairs:
        shutil.copy(img_path, det_out / 'images/train' / img_path.name)
        shutil.copy(label_path, det_out / 'labels/train' / label_path.name)
    for img_path, label_path in val_det_pairs:
        shutil.copy(img_path, det_out / 'images/val' / img_path.name)
        shutil.copy(label_path, det_out / 'labels/val' / label_path.name)

    # Create data.yaml for detection
    with open(det_out / 'data.yaml', 'w') as f:
        f.write(f"train: {det_out}/images/train\n")
        f.write(f"val: {det_out}/images/val\n")
        f.write("nc: 1\nnames: ['license_plate']\n") # Assuming class 0 is license_plate as per inspection

    # --- Character Recognition Preparation ---
    rec_images_src = DATASET_DIR / 'EALPR- Plates dataset'
    rec_labels_src = DATASET_DIR / 'EALPR- LP characters dataset/Characters Labeling'

    rec_pairs = []
    for label_path in sorted(list(rec_labels_src.glob('*.txt'))):
        stem = label_path.stem
        img_path_jpg = (rec_images_src / stem).with_suffix('.jpg')
        img_path_png = (rec_images_src / stem).with_suffix('.png')

        if img_path_jpg.exists():
            rec_pairs.append((img_path_jpg, label_path))
        elif img_path_png.exists():
            rec_pairs.append((img_path_png, label_path))
        # else:
        #     print(f"Warning: No image found for label {label_path.name} in {rec_images_src}")

    print(f"Found {len(rec_pairs)} image-label pairs for recognition.")
    if not rec_pairs:
        print("Error: No recognition image-label pairs found. Check dataset paths, file existence, and naming conventions.")
        return

    # Split recognition data
    split_idx_rec = int(len(rec_pairs) * 0.8)
    train_rec_pairs = rec_pairs[:split_idx_rec]
    val_rec_pairs = rec_pairs[split_idx_rec:]

    # Copy recognition data
    for img_path, label_path in train_rec_pairs:
        shutil.copy(img_path, rec_out / 'images/train' / img_path.name)
        shutil.copy(label_path, rec_out / 'labels/train' / label_path.name)
    for img_path, label_path in val_rec_pairs:
        shutil.copy(img_path, rec_out / 'images/val' / img_path.name)
        shutil.copy(label_path, rec_out / 'labels/val' / label_path.name)

    # Create data.yaml for recognition
    char_names = [str(i) for i in range(10)] + [chr(ord('A') + i) for i in range(26)] # 0-9, A-Z
    with open(rec_out / 'data.yaml', 'w') as f:
        f.write(f"train: {rec_out}/images/train\n")
        f.write(f"val: {rec_out}/images/val\n")
        f.write(f"nc: {len(char_names)}\nnames: {char_names}\n")

    print("Custom preparation complete. Generated data.yaml files and copied original labels for detection and recognition.")

try:
    simple_prepare()
except Exception as e:
    print(f"Preparation failed: {e}")

Starting custom dataset preparation...
Found 2031 image-label pairs for detection.
Found 1983 image-label pairs for recognition.
Custom preparation complete. Generated data.yaml files and copied original labels for detection and recognition.


### Inspecting Original Label Files

Before we can use the original labels, we need to understand their format. This cell will print the content of a sample label file from both the vehicle detection and character recognition datasets.

In [5]:
import os
from pathlib import Path

# Path to detection labels
det_labels_path = DATASET_DIR / 'EALPR Vechicles dataset/Vehicles Labeling'

# Path to recognition labels
rec_labels_path = DATASET_DIR / 'EALPR- LP characters dataset/Characters Labeling'

print(f"Inspecting Detection Labels from: {det_labels_path}")
if det_labels_path.exists():
    det_label_files = list(det_labels_path.glob('*.txt')) # Assuming .txt format for now, common for YOLO
    if det_label_files:
        sample_det_label = det_label_files[0]
        print(f"Sample detection label file: {sample_det_label}")
        with open(sample_det_label, 'r') as f:
            print(f"Content:\n{f.read()[:500]}... (truncated)") # Print first 500 chars
    else:
        print("No .txt label files found in detection labeling directory.")
else:
    print("Detection labels directory not found.")

print(f"\nInspecting Recognition Labels from: {rec_labels_path}")
if rec_labels_path.exists():
    rec_label_files = list(rec_labels_path.glob('*.txt')) # Assuming .txt format for now
    if rec_label_files:
        sample_rec_label = rec_label_files[0]
        print(f"Sample recognition label file: {sample_rec_label}")
        with open(sample_rec_label, 'r') as f:
            print(f"Content:\n{f.read()[:500]}... (truncated)") # Print first 500 chars
    else:
        print("No .txt label files found in recognition labeling directory.")
else:
    print("Recognition labels directory not found.")


Inspecting Detection Labels from: /content/Traffic-Serveillance-System/ai-service/dataset/EALPR Vechicles dataset/Vehicles Labeling
Sample detection label file: /content/Traffic-Serveillance-System/ai-service/dataset/EALPR Vechicles dataset/Vehicles Labeling/1151.txt
Content:
0 0.448611 0.514352 0.580556 0.308333
... (truncated)

Inspecting Recognition Labels from: /content/Traffic-Serveillance-System/ai-service/dataset/EALPR- LP characters dataset/Characters Labeling
Sample recognition label file: /content/Traffic-Serveillance-System/ai-service/dataset/EALPR- LP characters dataset/Characters Labeling/0604_license_plate_1.txt
Content:
19 0.2942176870748299 0.6575342465753424 0.08503401360544217 0.3561643835616438
2 0.6547619047619048 0.6643835616438356 0.12585034013605442 0.3424657534246575
2 0.7908163265306123 0.6643835616438356 0.12585034013605442 0.3561643835616438... (truncated)


In [6]:
# 5) Train detection model (YOLOv8)
from ultralytics import YOLO
import torch

# Detect device
device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

det_data_yaml = PROJECT_DIR / 'data/processed/detection/data.yaml'
det_model = YOLO('yolov8n.pt')

det_model.train(
    data=str(det_data_yaml),
    epochs=10,
    imgsz=640,
    device=device,
    project=str(DETECTION_ARTIFACTS),
    name='train',
    exist_ok=True,
)

print('Detection training complete.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cpu
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.9.0+cpu CPU (AMD EPYC 7B13)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Traffic-Serveillance-System/ai-service/data/processed/detection/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

In [8]:
# 6) Evaluate detection model and save prediction images
import shutil

device = 0 if torch.cuda.is_available() else 'cpu'
det_weights = DETECTION_ARTIFACTS / 'train' / 'weights' / 'best.pt'

# Ensure weights exist before proceeding
if not det_weights.exists():
    print(f'Warning: {det_weights} not found. Ensure training finished successfully.')
else:
    det_best = YOLO(str(det_weights))
    det_metrics = det_best.val(data=str(det_data_yaml), split='val', device=device)
    print('Detection mAP50:', getattr(getattr(det_metrics, 'box', None), 'map50', None))

    det_val_images = PROJECT_DIR / 'data/processed/detection/images/val'
    det_eval_out = EVAL_IMAGES_DIR / 'detection'
    det_eval_out.mkdir(parents=True, exist_ok=True)

    det_best.predict(
        source=str(det_val_images),
        device=device,
        save=True,
        conf=0.25,
        project=str(det_eval_out),
        name='preds',
        exist_ok=True,
    )
    print('Detection evaluation images saved to:', det_eval_out)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.9.0+cpu CPU (AMD EPYC 7B13)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3506.6±1361.7 MB/s, size: 97.6 KB)
val: Scanning /content/Traffic-Serveillance-System/ai-service/data/processed/detection/labels/val.cache... 406 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 407/407 213.4Mit/s 0.0s
val: /content/Traffic-Serveillance-System/ai-service/data/processed/detection/images/val/2018.jpg: ignoring corrupt image/label: image file is truncated (5 bytes not processed)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 1.4it/s 18.5s
                   all        406        410      0.997      0.993      0.995      0.915
Speed: 0.3ms preprocess, 36.5ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /content/Traffic-Serveillance-System/ai-service/runs/detect/val-2
Detection mAP50:

In [9]:
# 7) Train recognition model (YOLOv8 char detector)
device = 0 if torch.cuda.is_available() else 'cpu'
rec_data_yaml = PROJECT_DIR / 'data/processed/recognition/data.yaml'
rec_model = YOLO('yolov8n.pt')

rec_model.train(
    data=str(rec_data_yaml),
    epochs=10,
    imgsz=320,
    device=device,
    project=str(RECOGNITION_ARTIFACTS),
    name='train',
    exist_ok=True,
)

print('Recognition training complete.')

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.9.0+cpu CPU (AMD EPYC 7B13)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Traffic-Serveillance-System/ai-service/data/processed/recognition/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimi

In [10]:
# 8) Evaluate recognition model and save prediction images
device = 0 if torch.cuda.is_available() else 'cpu'
rec_weights = RECOGNITION_ARTIFACTS / 'train' / 'weights' / 'best.pt'

if not rec_weights.exists():
    print(f'Warning: {rec_weights} not found.')
else:
    rec_best = YOLO(str(rec_weights))
    rec_metrics = rec_best.val(data=str(rec_data_yaml), split='val', device=device)
    print('Recognition mAP50:', getattr(getattr(rec_metrics, 'box', None), 'map50', None))

    rec_val_images = PROJECT_DIR / 'data/processed/recognition/images/val'
    rec_eval_out = EVAL_IMAGES_DIR / 'recognition'
    rec_eval_out.mkdir(parents=True, exist_ok=True)

    rec_best.predict(
        source=str(rec_val_images),
        device=device,
        save=True,
        conf=0.25,
        project=str(rec_eval_out),
        name='preds',
        exist_ok=True,
    )
    print('Recognition evaluation images saved to:', rec_eval_out)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.9.0+cpu CPU (AMD EPYC 7B13)
Model summary (fused): 73 layers, 3,012,668 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1132.6±636.5 MB/s, size: 22.4 KB)
val: Scanning /content/Traffic-Serveillance-System/ai-service/data/processed/recognition/labels/val.cache... 397 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 397/397 208.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 5.2it/s 4.8s
                   all        397       2330      0.929      0.922      0.947      0.654
                     0         78         84      0.964      0.947      0.981      0.488
                     1         44         52          1      0.955      0.995      0.691
                     2         54         57      0.976          1      0.994      0.747
                     3         52         54      0.885          1      0.991      0.591
   

In [11]:
# 9) Copy best models to artifacts/models and summarize outputs
final_det = MODELS_ARTIFACTS / 'plate-detector.pt'
final_rec = MODELS_ARTIFACTS / 'plate-characters.pt'

shutil.copy2(det_weights, final_det)
shutil.copy2(rec_weights, final_rec)

print('Saved detector model:', final_det)
print('Saved recognizer model:', final_rec)
print('Artifacts root:', ARTIFACTS_DIR)

for p in [DETECTION_ARTIFACTS, RECOGNITION_ARTIFACTS, EVAL_IMAGES_DIR, MODELS_ARTIFACTS]:
    print('-', p)

Saved detector model: /content/Traffic-Serveillance-System/ai-service/artifacts/models/plate-detector.pt
Saved recognizer model: /content/Traffic-Serveillance-System/ai-service/artifacts/models/plate-characters.pt
Artifacts root: /content/Traffic-Serveillance-System/ai-service/artifacts
- /content/Traffic-Serveillance-System/ai-service/artifacts/detection
- /content/Traffic-Serveillance-System/ai-service/artifacts/recognition
- /content/Traffic-Serveillance-System/ai-service/artifacts/eval_images
- /content/Traffic-Serveillance-System/ai-service/artifacts/models


In [12]:
# 10) Optional: zip artifacts for easy download from Colab
archive_path = PROJECT_DIR / 'artifacts.zip'
if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(str(archive_path.with_suffix('')), 'zip', root_dir=ARTIFACTS_DIR)
print('Created:', archive_path)

# In Colab you can download with:
# from google.colab import files
# files.download(str(archive_path))

Created: /content/Traffic-Serveillance-System/ai-service/artifacts.zip
